# NGC 1068 FOV-cut significance comparison

This notebook applies each field-of-view cut consistently to the spacecraft orientation and the unbinned DC4 background. It then injects NGC 1068 using the cut orientation, bins the cut background in memory, and compares source and background counts. No intermediate products are saved.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astromodels import Cutoff_powerlaw
from threeML import Model, PointSource

from cosipy import SourceInjector, SpacecraftHistory
from cosipy.event_selection import GoodTimeInterval

%matplotlib inline

## Input files

In [ ]:
response_path = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)
orientation_path = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)
background_path = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Background/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits"
)
for path in (response_path, orientation_path, background_path):
    if not path.exists():
        raise FileNotFoundError(path)

## NGC 1068 source model

In [ ]:
source_name = "NGC1068"
source_coord = SkyCoord(l=172.104 * u.deg, b=-51.934 * u.deg, frame="galactic")

spectrum = Cutoff_powerlaw()
spectrum.K.value = 3.1e-1
spectrum.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum.piv.value = 1.0
spectrum.piv.unit = u.keV
spectrum.xc.value = 200.0
spectrum.xc.unit = u.keV
spectrum.index.value = -1.92

point_source = PointSource(
    source_name,
    l=source_coord.l.deg,
    b=source_coord.b.deg,
    spectral_shape=spectrum,
)
model = Model(point_source)

## Load the orientation and aggregate background times once

The background FITS table has about 169 million rows. Reading and binning all 11 columns for every FOV angle is unnecessarily expensive because only total background counts enter the significance. Instead, the code reads only `TimeTags` in bounded chunks and counts events per orientation interval once. Each pointing-cut GTI is then applied to the orientation and to those interval counts.

In [ ]:
orientation = SpacecraftHistory.open(orientation_path)

injector = SourceInjector(response_path=response_path)
total_livetime = orientation.cumulative_livetime().to_value(u.s)

orientation_edges_unix = orientation.obstime.unix
n_orientation_intervals = len(orientation.livetime)
background_counts_by_interval = np.zeros(n_orientation_intervals, dtype=np.int64)
chunk_size = 5_000_000

with fits.open(background_path, memmap=True) as hdul:
    events = hdul[1].data
    n_background_events = len(events)
    for start in range(0, n_background_events, chunk_size):
        stop = min(start + chunk_size, n_background_events)
        times = np.asarray(events["TimeTags"][start:stop])
        interval_index = np.searchsorted(orientation_edges_unix, times, side="right") - 1
        valid = (interval_index >= 0) & (interval_index < n_orientation_intervals)
        background_counts_by_interval += np.bincount(
            interval_index[valid], minlength=n_orientation_intervals
        )
        print(f"Background timing pass: {stop / n_background_events:.0%}", end="\r")

print()
print(f"Orientation samples: {len(orientation.obstime):,}")
print(f"Unbinned background events: {n_background_events:,}")
print(f"Total livetime: {total_livetime:,.1f} s")

## Scan the FOV cuts

The significance metric used for every cut is

$$\mathrm{significance} = \frac{S}{\sqrt{S+B}},$$

where $S$ and $B$ are the total counts in the cut source and background histograms.

In [ ]:
fov_angles = np.arange(30, 91, 5) * u.deg
earth_occ = True


def histogram_counts(histogram):
    contents = histogram.contents
    total = contents.sum()
    if hasattr(total, "compute"):
        total = total.compute()
    if hasattr(total, "todense"):
        total = total.todense()
    if hasattr(total, "value"):
        total = total.value
    return float(np.asarray(total).item())


rows = []

for max_offaxis in fov_angles:
    source_gti = GoodTimeInterval.from_pointing_cut(
        source_coord,
        orientation,
        max_offaxis,
        earth_occ=earth_occ,
    )
    if len(source_gti) == 0:
        print(f"Skipping {max_offaxis.value:.0f} deg: no selected exposure")
        continue

    cut_orientation = orientation.apply_gti(source_gti)
    cut_livetime = cut_orientation.cumulative_livetime().to_value(u.s)

    # Convert the merged GTIs back to orientation-bin indices so the same
    # selection can be applied to the pre-aggregated background counts.
    keep_interval = np.zeros(n_orientation_intervals, dtype=bool)
    for gti_start, gti_stop in source_gti:
        start_index = np.searchsorted(orientation_edges_unix, gti_start.unix, side="left")
        stop_index = np.searchsorted(orientation_edges_unix, gti_stop.unix, side="left")
        keep_interval[start_index:stop_index] = True

    background_counts = float(background_counts_by_interval[keep_interval].sum())
    if background_counts == 0 or cut_livetime <= 0:
        print(f"Skipping {max_offaxis.value:.0f} deg: no selected exposure/background")
        continue

    source_hist = injector.inject_model(
        model=model,
        orientation=cut_orientation,
        make_spectrum_plot=False,
        fluctuate=False,
        earth_occ=False,  # Earth occultation is already included in keep_interval.
    )

    source_counts = histogram_counts(source_hist)
    significance = source_counts / np.sqrt(source_counts + background_counts)

    angle = float(max_offaxis.to_value(u.deg))
    rows.append(
        {
            "fov_cut_deg": angle,
            "significance_s_over_sqrt_s_plus_b": significance,
        }
    )
    print(f"{angle:4.0f} deg: significance={significance:.3f} sigma")

results = pd.DataFrame(rows).sort_values("fov_cut_deg").reset_index(drop=True)
results

## Identify and visualize the best cut

In [ ]:
if results.empty:
    raise RuntimeError("No FOV cut produced usable source and background histograms.")

significance_column = "significance_s_over_sqrt_s_plus_b"
best_index = results[significance_column].idxmax()
best = results.loc[best_index]
best_angle = float(best["fov_cut_deg"])

print(f"Best FOV cut: {best_angle:.0f} deg")
print(f"S / sqrt(S + B): {best[significance_column]:.3f} sigma")

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ax.plot(results["fov_cut_deg"], results[significance_column], "o-", lw=2)
ax.axvline(best_angle, color="0.3", ls="--", label=f"Best")
ax.set_xlabel("Maximum off-axis angle [deg]")
ax.set_ylabel(r"$\sigma$")
ax.grid(alpha=0.3)
ax.legend(frameon=False)

The optimal angle above is based only on total source and background counts. For the final science analysis, verify the choice with the same likelihood, energy range, nuisance parameters, and response treatment used for the reported source significance.